# PyTorch Tutorial — Thesis-Focused
**Week 1 · Bachelor's Thesis: Benchmarking Tabular Data Augmentation Techniques for Deep Clustering**

Background: numpy ✓ · pandas ✓ · matplotlib ✓ · sklearn ✓ · PyTorch ← you are here

| § | Topic | Where you will use it |
|---|-------|-----------------------|
| 1 | Tensors & autograd | Foundation for every PyTorch operation |
| 2 | `Dataset` & `DataLoader` | Feed tabular rows to the encoder in batches |
| 3 | `nn.Module` — the encoder | The MLP `f(x)` that produces embeddings |
| 4 | Training loop & optimizer | Pre-train `f` with a loss function |
| ★ | **Capstone** | Full MLP classifier on Digits — typed from scratch |

**Thesis pipeline:**
```
numpy array (ColumnTransformer output)
  → TabularDataset / DataLoader     [§2]
  → MLPEncoder (nn.Module)          [§3]
  → training loop + Adam            [§4]
  → embeddings → k-means → NMI/ARI
```

**Canonical dimensions used throughout:** `input_dim=20` · `hidden_dim=256` · `embed_dim=64`

**Structure of each section:** concept + analogy → worked example → `# YOUR TURN:` exercise → solution.

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import numpy as np
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# ── CANONICAL THESIS DIMENSIONS ────────────────────────────────────────────
INPUT_DIM  = 20    # features per tabular row (simulated ColumnTransformer output)
HIDDEN_DIM = 256   # hidden layer width (matches SCARF paper architecture)
EMBED_DIM  = 64    # embedding dimension output by the encoder
BATCH_SIZE = 32
SEED       = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# ── SHARED SYNTHETIC TABULAR DATA ──────────────────────────────────────────
# Simulates the numpy array produced by ColumnTransformer in your thesis pipeline
rng      = np.random.default_rng(SEED)
X_synth  = rng.standard_normal((1000, INPUT_DIM)).astype(np.float32)  # (1000, 20)
y_synth  = rng.integers(0, 5, 1000).astype(np.int64)                  # 5 classes

print('X_synth shape:', X_synth.shape, ' dtype:', X_synth.dtype)
print('y_synth shape:', y_synth.shape, ' unique labels:', np.unique(y_synth))

X_synth shape: (1000, 20)  dtype: float32
y_synth shape: (1000,)  unique labels: [0 1 2 3 4]


---
## §1 Tensors and Autograd

### What is a tensor?

A tensor is PyTorch's multi-dimensional array — structurally identical to a numpy array.
The critical difference: a tensor can optionally **track every operation performed on it**
so that gradients can be computed automatically. That tracking ability is what makes training possible.

### What is autograd?

When you mark a tensor with `requires_grad=True`, PyTorch records every operation you apply
to it in a computation graph. Calling `.backward()` on a scalar result walks that graph in
reverse and deposits `∂result/∂tensor` into `tensor.grad`.

**Classical ML analogy — sklearn's SGD solver:**
```
sklearn SGDClassifier internally:    PyTorch autograd does:
  1. compute loss(w)                   1. build computation graph
  2. compute ∂loss/∂w by hand          2. loss.backward() — ∂loss/∂w auto-computed
  3. w ← w − lr · ∂loss/∂w            3. optimizer.step() — w ← w − lr · w.grad
```
Autograd replaces step 2 — it works for any computation, no matter how deep.

### The dtype trap

| Library | Default float dtype | What to use |
|---------|--------------------|--------------|
| numpy | `float64` | fine for sklearn |
| PyTorch | `float32` | **always use float32** — model weights are float32; mismatches crash |

Always pass `dtype=torch.float32` (or `.astype(np.float32)` before converting).

In [3]:
# ── §1 Worked example A: tensors ─────────────────────────────────────────────

# Create from a numpy array
arr   = np.array([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])           # shape (2, 3), dtype float64

t32 = torch.tensor(arr, dtype=torch.float32)  # always specify float32
print('tensor shape:', t32.shape)             # torch.Size([2, 3])
print('tensor dtype:', t32.dtype)             # torch.float32
print(t32)
print()

# Back to numpy
arr_back = t32.numpy()                        # shares memory — no copy
print('numpy shape:', arr_back.shape, ' dtype:', arr_back.dtype)
print()

# The dtype trap — what happens if you forget float32
t64 = torch.tensor(arr)                       # inherits float64 from numpy
print('without dtype=float32:', t64.dtype)    # torch.float64
print('→ model weights are float32; passing float64 tensors will crash at runtime')

tensor shape: torch.Size([2, 3])
tensor dtype: torch.float32
tensor([[1., 2., 3.],
        [4., 5., 6.]])

numpy shape: (2, 3)  dtype: float32

without dtype=float32: torch.float64
→ model weights are float32; passing float64 tensors will crash at runtime


In [4]:
# ── §1 Worked example B: autograd ────────────────────────────────────────────

# Simplest possible autograd example: y = x², dy/dx = 2x
x = torch.tensor(3.0, requires_grad=True)   # scalar, gradient tracking ON

y = x ** 2          # PyTorch records: "y was computed from x by squaring"

y.backward()        # walk the graph backward: compute dy/dx

print('x =', x.item())        # 3.0
print('y = x² =', y.item())   # 9.0
print('dy/dx =', x.grad.item())  # 2x = 2·3 = 6.0
print()

# ── The 3 lines you write in every training loop ─────────────────────────────
# 1. requires_grad=True  → mark the tensor to track (done automatically for nn.Module parameters)
# 2. loss.backward()     → compute ∂loss/∂(every requires_grad tensor)
# 3. tensor.grad         → read the gradient after backward()

# In training: PyTorch does this for ALL model weights simultaneously
w = torch.tensor(1.0, requires_grad=True)
loss = (w - 5.0) ** 2          # minimum at w=5
loss.backward()
print('Gradient-based update step:')
print(f'  w={w.item():.1f}  loss={loss.item():.1f}  grad={w.grad.item():.1f}')
print(f'  next w ≈ {w.item() - 0.1 * w.grad.item():.2f}  (moved toward minimum at 5.0)')

x = 3.0
y = x² = 9.0
dy/dx = 6.0

Gradient-based update step:
  w=1.0  loss=16.0  grad=-8.0
  next w ≈ 1.80  (moved toward minimum at 5.0)


In [5]:
# ── §1 YOUR TURN: exercise ───────────────────────────────────────────────────
# Create a tensor w = 2.0 with gradient tracking.
# Compute loss = (3w - 9)²
# Call .backward() and print w.grad
# Expected: w.grad = −18.0
#
# Verify by hand: d/dw (3w−9)² = 2·(3w−9)·3 = 6·(3w−9)
#                 at w=2: 6·(6−9) = 6·(−3) = −18  ✓

w = torch.tensor(2.0, requires_grad=True)                    # ← torch.tensor(2.0, requires_grad=True)
loss = (3*w - 9)**2                  # ← (3*w - 9)**2
loss.backward()                       # ← loss.backward()
print('w.grad =', w.grad)   # expected: tensor(-18.)

w.grad = tensor(-18.)


In [6]:
# ── §1 SOLUTION ──────────────────────────────────────────────────────────────

w = torch.tensor(2.0, requires_grad=True)
loss = (3 * w - 9) ** 2
loss.backward()
print('w.grad =', w.grad)   # tensor(-18.)
assert w.grad.item() == -18.0, f'Expected -18.0, got {w.grad.item()}'

w.grad = tensor(-18.)


---
## §2 Dataset and DataLoader

### The problem

Your preprocessed tabular data lives in a numpy array. PyTorch's training loop needs data
delivered as fixed-size float32 tensor batches, shuffled each epoch, with the last
incomplete batch handled cleanly. Writing that manually is error-prone.

`Dataset` and `DataLoader` solve this in two steps:

| Class | Your job | What it does |
|-------|----------|-------------|
| `Dataset` | Define `__len__` and `__getitem__` | Tells PyTorch how many rows there are and how to fetch row `i` |
| `DataLoader` | Pass your Dataset + batch size | Handles shuffling, batching, and the last partial batch |

**sklearn analogy — manual batching:**
```python
# What you would write without DataLoader:
idx = np.random.permutation(len(X))      # shuffle
for i in range(0, len(X), batch_size):   # batch
    X_batch = X[idx[i : i + batch_size]]
    y_batch = y[idx[i : i + batch_size]]

# What DataLoader does for you — same result, one line:
for X_batch, y_batch in loader:
    ...
```

### Thesis connection
```
ColumnTransformer → numpy array → TabularDataset → DataLoader → for X_batch in loader: model(X_batch)
                                  └── §2 defines this ──────────────────────────────────────────────┘
```

In [7]:
# ── §2 Worked example A: TabularDataset ──────────────────────────────────────

class TabularDataset(Dataset):
    """Wraps a preprocessed numpy array and integer labels as PyTorch tensors."""

    def __init__(self, X: np.ndarray, y: np.ndarray):
        self.X = torch.tensor(X, dtype=torch.float32)  # features: float32 — matches model weights
        self.y = torch.tensor(y, dtype=torch.long)     # labels: long (int64) — required by CrossEntropyLoss

    def __len__(self):
        return len(self.X)          # number of rows — DataLoader calls this to know when an epoch ends

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # DataLoader calls this for each index in a batch


# Create from the shared synthetic data
dataset = TabularDataset(X_synth, y_synth)

print('Dataset length:     ', len(dataset))       # 1000
print('Single row X shape: ', dataset[0][0].shape)  # torch.Size([20])
print('Single row y shape: ', dataset[0][1].shape)  # torch.Size([])  — scalar
print('X dtype:            ', dataset[0][0].dtype)  # torch.float32
print('y dtype:            ', dataset[0][1].dtype)  # torch.int64

Dataset length:      1000
Single row X shape:  torch.Size([20])
Single row y shape:  torch.Size([])
X dtype:             torch.float32
y dtype:             torch.int64


In [8]:
# ── §2 Worked example B: DataLoader and batch inspection ─────────────────────

loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f'Rows in dataset:   {len(dataset)}')
print(f'Batches per epoch: {len(loader)}')   # ceil(1000/32) = 32
print()

# Inspect the first batch
X_batch, y_batch = next(iter(loader))

print('First batch:')
print('  X_batch.shape:', X_batch.shape)   # torch.Size([32, 20])  — 32 rows × 20 features
print('  y_batch.shape:', y_batch.shape)   # torch.Size([32])       — 32 integer labels
print('  X_batch.dtype:', X_batch.dtype)   # torch.float32
print('  y_batch.dtype:', y_batch.dtype)   # torch.int64
print()
print('y_batch (first 8 labels):', y_batch[:8].tolist())

Rows in dataset:   1000
Batches per epoch: 32

First batch:
  X_batch.shape: torch.Size([32, 20])
  y_batch.shape: torch.Size([32])
  X_batch.dtype: torch.float32
  y_batch.dtype: torch.int64

y_batch (first 8 labels): [1, 2, 3, 4, 4, 3, 1, 2]


In [9]:
# ── §2 YOUR TURN: exercise ───────────────────────────────────────────────────
# Build a DataLoader from 500 random rows × 20 features (no labels needed here).
# Use TensorDataset — a built-in PyTorch class that wraps tensors directly
# without needing to write a custom Dataset class:
#
#   TensorDataset(tensor_a, tensor_b)  → dataset that returns (tensor_a[i], tensor_b[i])
#
# Steps:
#   1. Create X_ex as a torch.float32 tensor of shape (500, 20)
#   2. Wrap it in TensorDataset (use it as both X and a dummy copy: TensorDataset(X_ex, X_ex))
#   3. Create a DataLoader with batch_size=64
#   4. Print the shape of the first 3 batches

X_ex = torch.randn(500, 20)                         # ← torch.randn(500, 20)
ds_ex = TensorDataset(X_ex, X_ex )          # ← TensorDataset(X_ex, X_ex)
loader_ex = DataLoader(ds_ex, batch_size=64, shuffle=True)         # ← DataLoader(ds_ex, batch_size=64, shuffle=True)

print(f'Batches per epoch: {len(loader_ex)}')   # ceil(500/64) = 8
for i, (xb, _) in enumerate(loader_ex):
    print(f'  batch {i}: shape={xb.shape}')
    if i == 2:
        break

Batches per epoch: 8
  batch 0: shape=torch.Size([64, 20])
  batch 1: shape=torch.Size([64, 20])
  batch 2: shape=torch.Size([64, 20])


In [10]:
# ── §2 SOLUTION ──────────────────────────────────────────────────────────────

X_ex    = torch.randn(500, 20)
ds_ex   = TensorDataset(X_ex, X_ex)          # dummy (X, X) — only X shape matters here
loader_ex = DataLoader(ds_ex, batch_size=64, shuffle=True)

print(f'Batches per epoch: {len(loader_ex)}')   # 8
for i, (xb, _) in enumerate(loader_ex):
    print(f'  batch {i}: shape={xb.shape}')     # batches 0-6: (64, 20); batch 7: (52, 20)
    if i == 2:
        break
# Note: the last batch has 500 - 7×64 = 52 rows — DataLoader handles this automatically.

Batches per epoch: 8
  batch 0: shape=torch.Size([64, 20])
  batch 1: shape=torch.Size([64, 20])
  batch 2: shape=torch.Size([64, 20])


---
## §3 `nn.Module` — the Encoder

### The two-method structure

Every neural network in PyTorch is a class that inherits from `nn.Module`.
It must implement exactly two methods:

| Method | When it runs | What it does |
|--------|-------------|------------|
| `__init__` | Once, when you write `MLPEncoder(...)` | Defines and **registers** the layers |
| `forward` | Every time you write `model(x)` | Defines the data flow through the layers |

**sklearn analogy:**
- `__init__` is `KMeans(n_clusters=3)` — you declare the structure, nothing is computed yet
- `forward` is `kmeans.transform(X)` — you actually run the computation

**Why does PyTorch require this split?**
`__init__` makes the layers visible to PyTorch's internals: when you call
`model.parameters()`, PyTorch finds every weight tensor because it was registered in `__init__`.
The optimizer needs this list to know which tensors to update.
If you created layers inside `forward` instead, they would be invisible and never trained.

### Why no ReLU on the final layer?

ReLU clips all negative values to zero. An embedding that can only be positive loses half
its representational space. The InfoNCE loss (Week 4) works with ℓ₂-normalised embeddings
that must span the full hypersphere — including negative values.

### Thesis connection
```
DataLoader → X_batch (32, 20)
                 ↓  model(X_batch)
             z (32, 64)   ← this is f(x), the embedding used for k-means
```

In [11]:
# ── §3 Worked example A: define the encoder ──────────────────────────────────

class MLPEncoder(nn.Module):
    """3-layer MLP encoder. Maps a tabular row to a dense embedding vector."""

    def __init__(self, input_dim: int, hidden_dim: int, embed_dim: int):
        super().__init__()   # REQUIRED — registers this class as an nn.Module
                             # forgetting super().__init__() → AttributeError on first layer access

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),  # (20 → 256)  weight matrix: (256 × 20)
            nn.ReLU(),                          # max(0, x) element-wise — adds non-linearity
            nn.Linear(hidden_dim, hidden_dim),  # (256 → 256) weight matrix: (256 × 256)
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),   # (256 → 64)  weight matrix: (64 × 256)
            # No ReLU here — embeddings must be unconstrained to span the full hypersphere
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor: # what does this mean?
        return self.net(x)   # x: (batch, 20)  →  output: (batch, 64)


model = MLPEncoder(input_dim=INPUT_DIM, hidden_dim=HIDDEN_DIM, embed_dim=EMBED_DIM)
print(model)

MLPEncoder(
  (net): Sequential(
    (0): Linear(in_features=20, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=256, bias=True)
    (3): ReLU()
    (4): Linear(in_features=256, out_features=64, bias=True)
  )
)


In [12]:
# ── §3 Worked example B: forward pass + parameter count ──────────────────────

# Run a single forward pass with a fake batch
x_fake = torch.randn(BATCH_SIZE, INPUT_DIM)   # (32, 20) — fake batch of 32 tabular rows
z = model(x_fake)                              # calls model.forward(x_fake) internally

print('Input  shape:', x_fake.shape)   # torch.Size([32, 20])
print('Output shape:', z.shape)        # torch.Size([32, 64])  ← the embedding
assert z.shape == (BATCH_SIZE, EMBED_DIM), f'Expected ({BATCH_SIZE}, {EMBED_DIM}), got {z.shape}'
print('Shape assertion passed ✓')
print()

# model.parameters() — the optimizer needs this to know what to update
total_params = sum(p.numel() for p in model.parameters())
print(f'Trainable parameters: {total_params:,}')
print()
print('Layer-by-layer breakdown:')
for name, p in model.named_parameters():
    print(f'  {name:30s}  shape={str(p.shape):20s}  params={p.numel():,}')

Input  shape: torch.Size([32, 20])
Output shape: torch.Size([32, 64])
Shape assertion passed ✓

Trainable parameters: 87,616

Layer-by-layer breakdown:
  net.0.weight                    shape=torch.Size([256, 20])  params=5,120
  net.0.bias                      shape=torch.Size([256])     params=256
  net.2.weight                    shape=torch.Size([256, 256])  params=65,536
  net.2.bias                      shape=torch.Size([256])     params=256
  net.4.weight                    shape=torch.Size([64, 256])  params=16,384
  net.4.bias                      shape=torch.Size([64])      params=64


In [13]:
# ── §3 YOUR TURN: exercise ───────────────────────────────────────────────────
# Define MLPEncoder4 — identical to MLPEncoder but with a 4th hidden layer:
#   Linear(20→256) → ReLU → Linear(256→256) → ReLU → Linear(256→256) → ReLU → Linear(256→64)
#
# Then:
#   1. Instantiate it
#   2. Run a forward pass with x_fake (shape: (32, 20))
#   3. Assert the output shape is still (32, 64)
#   4. Print the total parameter count — compare to the 3-layer version above

class MLPEncoder4(nn.Module):
    def __init__(self, input_dim, hidden_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),   # ← your 4-layer architecture here
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim)
        )
    def forward(self, x):
        return self.net(x)


model4 = MLPEncoder4(INPUT_DIM, HIDDEN_DIM, EMBED_DIM)
z4 = model4(x_fake)
print('Output shape:', z4.shape)                                          # (32, 64)
assert z4.shape == (BATCH_SIZE, EMBED_DIM)
print('Params (4-layer):', sum(p.numel() for p in model4.parameters()), '\n'
      'Params (3-layer):', sum(p.numel() for p in model.parameters()))

Output shape: torch.Size([32, 64])
Params (4-layer): 153408 
Params (3-layer): 87616


In [14]:
# ── §3 SOLUTION ──────────────────────────────────────────────────────────────

class MLPEncoder4(nn.Module):
    def __init__(self, input_dim, hidden_dim, embed_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),  # extra layer
            nn.ReLU(),
            nn.Linear(hidden_dim, embed_dim),   # still no final ReLU
        )
    def forward(self, x):
        return self.net(x)


model4 = MLPEncoder4(INPUT_DIM, HIDDEN_DIM, EMBED_DIM)
z4 = model4(x_fake)
print('Output shape:', z4.shape)
assert z4.shape == (BATCH_SIZE, EMBED_DIM)
print(f'Params 3-layer: {sum(p.numel() for p in model.parameters()):,}')
print(f'Params 4-layer: {sum(p.numel() for p in model4.parameters()):,}')
# The extra 256×256 layer adds 256*256 + 256 = 65,792 parameters

Output shape: torch.Size([32, 64])
Params 3-layer: 87,616
Params 4-layer: 153,408


---
## §4 Training Loop and Optimizer

### The canonical 5-line loop

```python
optimizer.zero_grad()       # 1. clear accumulated gradients from the previous batch
output = model(X_batch)     # 2. forward pass — compute predictions
loss = criterion(output, T) # 3. compute scalar loss
loss.backward()             # 4. compute ∂loss/∂(every parameter)
optimizer.step()            # 5. update every parameter: w ← w − lr · w.grad
```

**What breaks if you skip each line:**

| Skipped | What breaks |
|---------|-------------|
| `zero_grad()` | PyTorch **accumulates** gradients by default — gradients from batch 1 remain when batch 2 runs. Weights update as if trained on multiple batches at once. Loss oscillates wildly. **This is the #1 beginner mistake.** |
| `loss.backward()` | No gradients are computed — `w.grad` is `None`. `optimizer.step()` does nothing silently. Loss never decreases. |
| `optimizer.step()` | Gradients computed but weights never changed. Loss never decreases. |

### About `loss.item()`

`loss` is a single-element tensor. `.item()` converts it to a plain Python float so you
can accumulate it in a regular variable. Never accumulate the tensor itself — it holds
the entire computation graph in memory.

### Thesis connection

In Week 4 you will replace `nn.MSELoss` with the InfoNCE contrastive loss.
Every other line in this loop stays identical — only `criterion` changes.

In [15]:
# ── §4 Worked example: full training loop ────────────────────────────────────
#
# Task: train MLPEncoder to map each input row to a fixed target embedding.
# Loss: MSELoss(model(x), target_embedding)
# In Week 4: target_embedding is replaced by the corrupted view's embedding,
#            MSELoss is replaced by InfoNCE — everything else stays the same.

# ── Dataset: (X_synth, T_synth) — X are inputs, T are synthetic target embeddings
T_synth = rng.standard_normal((1000, EMBED_DIM)).astype(np.float32)  # (1000, 64)

X_t = torch.tensor(X_synth, dtype=torch.float32)
T_t = torch.tensor(T_synth, dtype=torch.float32)
train_loader = DataLoader(TensorDataset(X_t, T_t), batch_size=BATCH_SIZE, shuffle=True)

# ── Model, loss, optimizer
encoder   = MLPEncoder(INPUT_DIM, HIDDEN_DIM, EMBED_DIM)
criterion = nn.MSELoss()                                  # ← Week 4: replace with InfoNCE
optimizer = torch.optim.Adam(encoder.parameters(), lr=0.001)  # Adam default lr=0.001

# ── 10-epoch training loop
for epoch in range(10):
    encoder.train()          # sets training mode (good habit; matters when using Dropout/BatchNorm)
    total_loss = 0.0

    for X_batch, T_batch in train_loader:
        optimizer.zero_grad()              # 1. clear old gradients — MUST come first
        z = encoder(X_batch)               # 2. forward pass: (32, 20) → (32, 64)
        loss = criterion(z, T_batch)       # 3. MSE between embedding and target
        loss.backward()                    # 4. compute all gradients
        optimizer.step()                   # 5. update weights
        total_loss += loss.item()          # .item() → plain float, avoids memory leak

    avg = total_loss / len(train_loader)
    print(f'Epoch {epoch+1:2d}/10  loss={avg:.4f}')

# Loss should decrease each epoch — the encoder is learning to predict T_synth
# In Week 4: replace nn.MSELoss with InfoNCE — every other line stays the same.

Epoch  1/10  loss=1.0108
Epoch  2/10  loss=0.9951
Epoch  3/10  loss=0.9797
Epoch  4/10  loss=0.9691
Epoch  5/10  loss=0.9531
Epoch  6/10  loss=0.9366
Epoch  7/10  loss=0.9194
Epoch  8/10  loss=0.9017
Epoch  9/10  loss=0.8867
Epoch 10/10  loss=0.8665


In [16]:
# ── §4 YOUR TURN: exercise ───────────────────────────────────────────────────
# Copy the training loop above. Change only ONE thing: set lr=0.1 instead of 0.001.
# Run it and observe what happens to the loss.
#
# Then answer in a comment: does the loss decrease smoothly, oscillate, or diverge?
# Why does a high learning rate cause this behaviour?

encoder_ex = MLPEncoder(INPUT_DIM, HIDDEN_DIM, EMBED_DIM)
optimizer_ex = torch.optim.Adam(encoder_ex.parameters(), lr=0.1)   # ← 0.1

for epoch in range(10):
    encoder_ex.train()
    total_loss = 0.0
    for X_batch, T_batch in train_loader:
        optimizer_ex.zero_grad()   # ← same 5 lines as above
        z = encoder_ex(X_batch)
        loss = criterion(z, T_batch)
        loss.backward()
        optimizer_ex.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1:2d}/10  loss={total_loss / len(train_loader):.4f}')

# Answer: lr=0.1 causes ...

Epoch  1/10  loss=8.6219
Epoch  2/10  loss=1.1005
Epoch  3/10  loss=1.0072
Epoch  4/10  loss=1.0415
Epoch  5/10  loss=1.0083
Epoch  6/10  loss=1.0050
Epoch  7/10  loss=1.0061
Epoch  8/10  loss=1.0046
Epoch  9/10  loss=1.0090
Epoch 10/10  loss=1.0059


In [17]:
# ── §4 SOLUTION ──────────────────────────────────────────────────────────────

encoder_ex   = MLPEncoder(INPUT_DIM, HIDDEN_DIM, EMBED_DIM)
optimizer_ex = torch.optim.Adam(encoder_ex.parameters(), lr=0.1)  # lr=0.1 — too large

for epoch in range(10):
    encoder_ex.train()
    total_loss = 0.0
    for X_batch, T_batch in train_loader:
        optimizer_ex.zero_grad()
        z = encoder_ex(X_batch)
        loss = criterion(z, T_batch)
        loss.backward()
        optimizer_ex.step()
        total_loss += loss.item()
    print(f'Epoch {epoch+1:2d}/10  loss={total_loss / len(train_loader):.4f}')

# Answer: with lr=0.1, Adam takes steps that are 100× larger than with lr=0.001.
# The weights overshoot the minimum each step — loss oscillates or diverges
# instead of decreasing smoothly.
# Rule of thumb for Adam: start with lr=0.001 (the default) and only change it
# if training is clearly too slow.

Epoch  1/10  loss=19.1190
Epoch  2/10  loss=1.0674
Epoch  3/10  loss=1.0117
Epoch  4/10  loss=1.0045
Epoch  5/10  loss=1.0060
Epoch  6/10  loss=1.0086
Epoch  7/10  loss=1.0069
Epoch  8/10  loss=1.0060
Epoch  9/10  loss=1.0039
Epoch 10/10  loss=1.0070


---
## ★ Capstone — MLP Classifier on Digits

### Goal

Type this section from scratch — without copying from above.
Every component has been introduced in §1–§4. Wire them together.

### Pipeline
```
load_digits() → StandardScaler → train_test_split
  → TabularDataset → DataLoader
  → MLPClassifier(input_dim=64, hidden_dim=256, n_classes=10)
  → 20-epoch training loop with CrossEntropyLoss
  → model.eval() + torch.no_grad() + logits.argmax(dim=1)
  → print test accuracy  (target: ≥ 0.93)
```

### New pieces (not in §1–§4)

| New item | Notes |
|----------|-------|
| `nn.CrossEntropyLoss` | Takes raw logits `(batch, n_classes)` and integer labels `(batch,)`. Do **not** add softmax to your model — CrossEntropyLoss includes it internally. |
| `model.eval()` | Puts the model in evaluation mode. Symmetric with `model.train()`. Matters for Dropout/BatchNorm; good habit always. |
| `torch.no_grad()` | Context manager — disables autograd during inference. Saves memory and speeds up the forward pass. Always use during evaluation. |
| `logits.argmax(dim=1)` | Returns the index of the highest value along dimension 1 (the class dimension). Shape `(batch, n_classes)` → `(batch,)`. That index is the predicted class. |

In [18]:
# ── ★ Capstone ────────────────────────────────────────────────────────────────

# ── 1. Load and preprocess ───────────────────────────────────────────────────
digits = load_digits()
X_d, y_d = digits.data, digits.target   # (1797, 64) float64, (1797,) int

scaler  = StandardScaler()
X_d_sc  = scaler.fit_transform(X_d).astype(np.float32)   # float32 for PyTorch

X_tr, X_te, y_tr, y_te = train_test_split(
    X_d_sc, y_d, test_size=0.2, random_state=SEED
)

print('Train:', X_tr.shape, '  Test:', X_te.shape)

# ── 2. Dataset and DataLoader ────────────────────────────────────────────────
train_ds = TabularDataset(X_tr, y_tr)
test_ds  = TabularDataset(X_te, y_te)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False)   # no shuffle at eval time

# ── 3. Model ─────────────────────────────────────────────────────────────────
class MLPClassifier(nn.Module):
    """3-layer MLP for classification. Output: raw logits, one per class."""
    def __init__(self, input_dim: int, hidden_dim: int, n_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes),  # output: (batch, 10) logits — no softmax
        )
    def forward(self, x):
        return self.net(x)

clf       = MLPClassifier(input_dim=64, hidden_dim=256, n_classes=10)
criterion = nn.CrossEntropyLoss()                       # expects raw logits, not softmax
optimizer = torch.optim.Adam(clf.parameters(), lr=0.001)

# ── 4. Training loop ─────────────────────────────────────────────────────────
for epoch in range(20):
    clf.train()                          # training mode
    total_loss = 0.0

    for X_batch, y_batch in train_dl:
        optimizer.zero_grad()            # 1. clear gradients
        logits = clf(X_batch)            # 2. forward: (batch, 64) → (batch, 10)
        loss = criterion(logits, y_batch)# 3. cross-entropy between logits and true labels
        loss.backward()                  # 4. compute gradients
        optimizer.step()                 # 5. update weights
        total_loss += loss.item()

    print(f'Epoch {epoch+1:2d}/20  loss={total_loss / len(train_dl):.4f}')

# ── 5. Evaluation ────────────────────────────────────────────────────────────
clf.eval()                               # evaluation mode — disables dropout/batchnorm randomness
correct, total = 0, 0

with torch.no_grad():                    # disable autograd — faster, less memory
    for X_batch, y_batch in test_dl:
        logits = clf(X_batch)            # (batch, 10) raw scores
        preds  = logits.argmax(dim=1)    # (batch,) — index of highest score = predicted class
        correct += (preds == y_batch).sum().item()
        total   += len(y_batch)

accuracy = correct / total
print(f'\nTest accuracy: {accuracy:.3f}   (target: ≥ 0.93)')
assert accuracy >= 0.93, f'Expected ≥ 0.93, got {accuracy:.3f}'

Train: (1437, 64)   Test: (360, 64)
Epoch  1/20  loss=1.1406
Epoch  2/20  loss=0.1951
Epoch  3/20  loss=0.0753
Epoch  4/20  loss=0.0464
Epoch  5/20  loss=0.0264
Epoch  6/20  loss=0.0143
Epoch  7/20  loss=0.0099
Epoch  8/20  loss=0.0069
Epoch  9/20  loss=0.0055
Epoch 10/20  loss=0.0040
Epoch 11/20  loss=0.0032
Epoch 12/20  loss=0.0027
Epoch 13/20  loss=0.0022
Epoch 14/20  loss=0.0019
Epoch 15/20  loss=0.0016
Epoch 16/20  loss=0.0014
Epoch 17/20  loss=0.0012
Epoch 18/20  loss=0.0011
Epoch 19/20  loss=0.0010
Epoch 20/20  loss=0.0008

Test accuracy: 0.983   (target: ≥ 0.93)


---
## Quick Reference Card

| Call | What it does |
|------|--------------|
| `torch.tensor(arr, dtype=torch.float32)` | numpy array → float32 tensor |
| `t.numpy()` | tensor → numpy array (shares memory) |
| `torch.tensor(x, requires_grad=True)` | enable gradient tracking on a tensor |
| `loss.backward()` | compute ∂loss/∂(all requires_grad tensors) |
| `tensor.grad` | read gradient after `.backward()` |
| `loss.item()` | scalar tensor → Python float (use inside training loop) |
| `TabularDataset(X, y)` | wrap numpy arrays as a `Dataset` |
| `TensorDataset(t1, t2)` | wrap tensors directly as a `Dataset` (no custom class needed) |
| `DataLoader(ds, batch_size=32, shuffle=True)` | batched, shuffled iterator over a Dataset |
| `for X_b, y_b in loader:` | iterate over all batches in one epoch |
| `class Net(nn.Module)` | define a network; must call `super().__init__()` |
| `nn.Sequential(layer1, layer2, ...)` | chain layers; `forward` calls them in order |
| `nn.Linear(in, out)` | fully-connected layer: `output = W·x + b` |
| `nn.ReLU()` | element-wise `max(0, x)` |
| `model(x)` | run forward pass (calls `model.forward(x)`) |
| `model.parameters()` | iterator over all trainable weight tensors |
| `sum(p.numel() for p in model.parameters())` | total parameter count |
| `torch.optim.Adam(model.parameters(), lr=0.001)` | Adam optimiser with default lr |
| `optimizer.zero_grad()` | clear accumulated gradients — call before every backward |
| `optimizer.step()` | apply gradients: `w ← w − lr · w.grad` |
| `nn.MSELoss()` | mean squared error — placeholder until InfoNCE (Week 4) |
| `nn.CrossEntropyLoss()` | classification loss — expects raw logits, not softmax output |
| `logits.argmax(dim=1)` | predicted class index from logits: `(batch, C)` → `(batch,)` |
| `model.train()` | set training mode (enable dropout/batchnorm) |
| `model.eval()` | set evaluation mode (disable dropout/batchnorm) |
| `with torch.no_grad():` | disable autograd during inference — faster, less memory |